In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import Descriptors

from sklearn.model_selection import train_test_split

from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, roc_curve, roc_auc_score
from sklearn.model_selection import GridSearchCV

from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline

In [2]:
dataset_with_violations = pd.read_csv('datasets/dataset_with_number_of_violatons.csv')

In [3]:
dataset_with_violations

,0,Acetyl-DL-carnitine,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C,203.24,0.4,0.1,4,66.4,5,0.2,0.3,0.5
0,1,[2-(Acetyloxy)-3-carboxypropyl]trimethylazanium,CC(=O)OC(CC(=O)O)C[N+](C)(C)C,204.24,-0.3,1,4,63.6,6,1,0,0
1,2,1-Amino-2-propanol,CC(CN)O,75.11,-1.0,2,2,46.3,1,1,0,2
2,3,Dinitrochlorobenzene,C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl,202.55,2.3,0,4,91.6,0,0,0,0
3,4,9-Ethyladenine,CCN1C=NC2=C(N=CN=C21)N,163.18,0.2,1,4,69.6,1,0,0,1
4,5,"2,3-Dihydroxy-3-methylpentanoic acid",CCC(C)(C(C(=O)O)O)O,148.16,-0.4,3,4,77.8,3,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
305244,331893,Cholesteryl Petroselinate,CCCCCCCCCCC/C=CCCCCC(=O)O[C@H]1CC[C@@]2([C@H]3...,651.10,16.6,0,2,26.3,22,2,2,3
305245,331895,"(2S,3S,4R,5R,6S)-6-[[(1S,19R,22R,37R,40R,52S)-...",CC(C)CCCCCCCCC(=O)N[C@@H]1[C@H]([C@@H]([C@H](O...,1816.70,3.8,21,30,573.0,22,3,5,5
305246,331896,"(2aR,3S,4S,4aR,5S,7aS,8S,10R,10aS,10bR)-10-(Ac...",C/C=C(C)/C(=O)O[C@H]1C[C@H]([C@]2(CO[C@@H]3[C@...,720.70,-0.6,3,16,215.0,10,3,1,3
305247,331897,"(4R,4aR,7S,7aR,12bS)-7-hydroxy-10-methoxy-3-me...",CN1CC[C@]23[C@@H]4[C@H]1CC5=C2C(=CC(=C5C(=O)O)...,343.40,-1.6,2,6,79.2,2,1,0,0


In [4]:
dataset_with_violations.drop(columns=['0'], inplace=True)

In [5]:
dataset_with_violations

,Acetyl-DL-carnitine,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C,203.24,0.4,0.1,4,66.4,5,0.2,0.3,0.5
0,[2-(Acetyloxy)-3-carboxypropyl]trimethylazanium,CC(=O)OC(CC(=O)O)C[N+](C)(C)C,204.24,-0.3,1,4,63.6,6,1,0,0
1,1-Amino-2-propanol,CC(CN)O,75.11,-1.0,2,2,46.3,1,1,0,2
2,Dinitrochlorobenzene,C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl,202.55,2.3,0,4,91.6,0,0,0,0
3,9-Ethyladenine,CCN1C=NC2=C(N=CN=C21)N,163.18,0.2,1,4,69.6,1,0,0,1
4,"2,3-Dihydroxy-3-methylpentanoic acid",CCC(C)(C(C(=O)O)O)O,148.16,-0.4,3,4,77.8,3,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...
305244,Cholesteryl Petroselinate,CCCCCCCCCCC/C=CCCCCC(=O)O[C@H]1CC[C@@]2([C@H]3...,651.10,16.6,0,2,26.3,22,2,2,3
305245,"(2S,3S,4R,5R,6S)-6-[[(1S,19R,22R,37R,40R,52S)-...",CC(C)CCCCCCCCC(=O)N[C@@H]1[C@H]([C@@H]([C@H](O...,1816.70,3.8,21,30,573.0,22,3,5,5
305246,"(2aR,3S,4S,4aR,5S,7aS,8S,10R,10aS,10bR)-10-(Ac...",C/C=C(C)/C(=O)O[C@H]1C[C@H]([C@]2(CO[C@@H]3[C@...,720.70,-0.6,3,16,215.0,10,3,1,3
305247,"(4R,4aR,7S,7aR,12bS)-7-hydroxy-10-methoxy-3-me...",CN1CC[C@]23[C@@H]4[C@H]1CC5=C2C(=CC(=C5C(=O)O)...,343.40,-1.6,2,6,79.2,2,1,0,0


In [7]:
dataset_with_violations.columns = ['cmpdname', 'smiles', 'mw', 'xlogp', 'hbonddonor', 'hbondacc', 'polararea', 'rotbonds', 'nviolations_lipinski_rule', 'nviolations_bro_rule', 'nviolations_muegge_rule']

In [8]:
dataset_with_violations

,cmpdname,smiles,mw,xlogp,hbonddonor,hbondacc,polararea,rotbonds,nviolations_lipinski_rule,nviolations_bro_rule,nviolations_muegge_rule
0,[2-(Acetyloxy)-3-carboxypropyl]trimethylazanium,CC(=O)OC(CC(=O)O)C[N+](C)(C)C,204.24,-0.3,1,4,63.6,6,1,0,0
1,1-Amino-2-propanol,CC(CN)O,75.11,-1.0,2,2,46.3,1,1,0,2
2,Dinitrochlorobenzene,C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl,202.55,2.3,0,4,91.6,0,0,0,0
3,9-Ethyladenine,CCN1C=NC2=C(N=CN=C21)N,163.18,0.2,1,4,69.6,1,0,0,1
4,"2,3-Dihydroxy-3-methylpentanoic acid",CCC(C)(C(C(=O)O)O)O,148.16,-0.4,3,4,77.8,3,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...
305244,Cholesteryl Petroselinate,CCCCCCCCCCC/C=CCCCCC(=O)O[C@H]1CC[C@@]2([C@H]3...,651.10,16.6,0,2,26.3,22,2,2,3
305245,"(2S,3S,4R,5R,6S)-6-[[(1S,19R,22R,37R,40R,52S)-...",CC(C)CCCCCCCCC(=O)N[C@@H]1[C@H]([C@@H]([C@H](O...,1816.70,3.8,21,30,573.0,22,3,5,5
305246,"(2aR,3S,4S,4aR,5S,7aS,8S,10R,10aS,10bR)-10-(Ac...",C/C=C(C)/C(=O)O[C@H]1C[C@H]([C@]2(CO[C@@H]3[C@...,720.70,-0.6,3,16,215.0,10,3,1,3
305247,"(4R,4aR,7S,7aR,12bS)-7-hydroxy-10-methoxy-3-me...",CN1CC[C@]23[C@@H]4[C@H]1CC5=C2C(=CC(=C5C(=O)O)...,343.40,-1.6,2,6,79.2,2,1,0,0


In [9]:
stop

NameError: name 'stop' is not defined

In [ ]:
molecule_ml1 = Chem.MolFromSmiles('N[C@@H](CCCNC(=N)N)C(=O)NCC(=O)N[C@@H](CC(=O)O)C(=O)O')
molecule_ml2 = Chem.MolFromSmiles('NC(CCNC(=N)N)C(=O)NCC(=O)N[C@@H](CC(=O)O)C(=O)O')
molecule_ml3 = Chem.MolFromSmiles('NC(CNC(=N)N)C(=O)NCC(=O)N[C@@H](CC(=O)O)C(=O)O')
molecule_ml4 = Chem.MolFromSmiles('N[C@@H](CCONC(=N)N)C(=O)OCC(=O)N[C@@H](CC(=O)O)C(=O)O')
molecule_ml5 = Chem.MolFromSmiles('NCCNC(=O)[C@H](CC(=O)O)NC(=O)CNC(=O)[C@@H](N)CCCNC(=N)N')
molecule_ml6 = Chem.MolFromSmiles('NCCCNC(=O)[C@H](CC(=O)O)NC(=O)CNC(=O)[C@@H](N)CCCNC(=N)N')
molecule_ml7 = Chem.MolFromSmiles('NCCNC(=O)[C@H](CC(=O)O)NC(=O)CNC(=O)C(N)CCNC(=N)N')
molecule_ml8 = Chem.MolFromSmiles('NCCCNC(=O)[C@H](CC(=O)O)NC(=O)CNC(=O)C(N)CCNC(=N)N')
molecule_ml9 = Chem.MolFromSmiles('NCCNC(=O)[C@H](CC(=O)O)NC(=O)CNC(=O)C(N)CNC(=N)N')
molecule_ml10 = Chem.MolFromSmiles('NCCCNC(=O)[C@H](CC(=O)O)NC(=O)CNC(=O)C(N)CNC(=N)N')
molecule_ml11 = Chem.MolFromSmiles('NCCNC(=O)[C@H](CC(=O)O)NC(=O)COC(=O)[C@@H](N)CCONC(=N)N')
molecule_ml12 = Chem.MolFromSmiles('NCCCNC(=O)[C@H](CC(=O)O)NC(=O)COC(=O)[C@@H](N)CCONC(=N)N')
molecule_ml13 = Chem.MolFromSmiles('CC[C@H](C)[C@@H](NC(=O)[C@@H]1CCCN1C(=O)[C@@H](NC(=O)[C@H](C)N)C(C)C)C(=O)O')
molecule_ml14 = Chem.MolFromSmiles('CC[C@H](C)[C@@H](NC(=O)[C@@H]1CCCN1C(=O)[C@H](Cc2ccc(F)cc2)NC(=O)[C@H](C)N)C(=O)O')
molecule_ml15 = Chem.MolFromSmiles('C[C@H](N)C(=O)N[C@@H](Cc1ccc(F)cc1)C(=O)N2CCC[C@H]2C(=O)N[C@H](Cc3ccc(F)cc3)C(=O)O')
molecule_ml16 = Chem.MolFromSmiles('CC(C)[C@H](NC(=O)[C@H](C)N)C(=O)N1CCC[C@H]1C(=O)N[C@H](Cc2ccc(F)cc2)C(=O)O')
molecule_ml17 = Chem.MolFromSmiles('CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccc(O)cc1)NC(=O)C2CCCN2C(=O)[C@H](CCCNC(=N)N)NC(=O)[C@@H](N)CCCNC(=N)N)C(=O)N[C@@H](CC(C)C)C(=O)O')
molecule_ml18 = Chem.MolFromSmiles('CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccc(O)cc1)NC(=O)C2CCCN2C(=O)[C@H](CCONC(=N)N)NC(=O)[C@@H](N)CCCCN)C(=O)N[C@@H](CC(C)C)C(=O)O')
molecule_ml19 = Chem.MolFromSmiles('CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccc(O)cc1)NC(=O)C2CCCN2C(=O)[C@H](CCCCN)NC(=O)[C@@H](N)CCONC(=N)N)C(=O)N[C@@H](CC(C)C)C(=O)O')
molecule_ml20 = Chem.MolFromSmiles('CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccc(O)cc1)NC(=O)C2CCCN2C(=O)[C@H](CCONC(=N)N)NC(=O)[C@@H](N)CCONC(=N)N)C(=O)N[C@@H](CC(C)C)C(=O)O')
molecule_ml21 = Chem.MolFromSmiles('CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccc(O)cc1)NC(=O)C2CCCN2C(=O)[C@H](CCONC(=N)N)NC(=O)[C@@H](N)CCCNC(=N)N)C(=O)N[C@@H](CC(C)C)C(=O)O')
molecule_ml22 = Chem.MolFromSmiles('CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccc(O)cc1)NC(=O)C2CCCN2C(=O)[C@H](CCCNC(=N)N)NC(=O)[C@@H](N)CCONC(=N)N)C(=O)N[C@@H](CC(C)C)C(=O)O')
molecule_ml23 = Chem.MolFromSmiles('CC[C@@H](C)[C@H](NC(=O)[C@H](Cc1ccc(F)cc1)NC(=O)C2CCCN2C(=O)[C@H](CCCNC(=N)N)NC(=O)[C@@H](N)CCONC(=N)N)C(=O)N[C@@H](CC(C)C)C(=O)O')
molecule_ml24 = Chem.MolFromSmiles('CCC(C)[C@H](NC(=O)[C@@H]1CCCN1C(=O)[C@@H](NC(=O)[C@H](C)N)C(C)C)C(=O)N[C@@H](CCCNC(=N)N)C(=O)N[C@@H](CCCNC(=N)N)C(=O)N[C@@H](CCCNC(=N)N)C(=O)N[C@@H](CCCNC(=N)N)C(=O)N[C@@H](CCCNC(=N)N)C(=O)N[C@@H](CCCNC(=N)N)C(=O)O')
molecule_ml25 = Chem.MolFromSmiles('CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccc(O)cc1)NC(=O)C2CCCN2C(=O)[C@H](CCONC(=N)N)NC(=O)[C@H](CCONC(=N)N)NC(=O)[C@@H](NC(=O)[C@@H]3CCCN3C(=O)[C@@H](NC(=O)[C@H](C)N)C(C)C)C(C)CC)C(=O)N[C@@H](CC(C)C)C(=O)O')
molecule_ml26 = Chem.MolFromSmiles('CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccc(O)cc1)NC(=O)C2CCCN2C(=O)[C@H](CCCNC(=N)N)NC(=O)[C@H](CCCNC(=N)N)NC(=O)[C@@H](NC(=O)[C@@H]3CCCN3C(=O)[C@@H](NC(=O)[C@H](C)N)C(C)C)C(C)CC)C(=O)N[C@@H](CC(C)C)C(=O)O')

In [ ]:
descriptors_ml1 = Descriptors.CalcMolDescriptors(molecule_ml1)
descriptors_ml2 = Descriptors.CalcMolDescriptors(molecule_ml2)
descriptors_ml3 = Descriptors.CalcMolDescriptors(molecule_ml3)
descriptors_ml4 = Descriptors.CalcMolDescriptors(molecule_ml4)
descriptors_ml5 = Descriptors.CalcMolDescriptors(molecule_ml5)
descriptors_ml6 = Descriptors.CalcMolDescriptors(molecule_ml6)
descriptors_ml7 = Descriptors.CalcMolDescriptors(molecule_ml7)
descriptors_ml8 = Descriptors.CalcMolDescriptors(molecule_ml8)
descriptors_ml9 = Descriptors.CalcMolDescriptors(molecule_ml9)
descriptors_ml10 = Descriptors.CalcMolDescriptors(molecule_ml10)
descriptors_ml11 = Descriptors.CalcMolDescriptors(molecule_ml11)
descriptors_ml12 = Descriptors.CalcMolDescriptors(molecule_ml12)
descriptors_ml13 = Descriptors.CalcMolDescriptors(molecule_ml13)
descriptors_ml14 = Descriptors.CalcMolDescriptors(molecule_ml14)
descriptors_ml15 = Descriptors.CalcMolDescriptors(molecule_ml15)
descriptors_ml16 = Descriptors.CalcMolDescriptors(molecule_ml16)
descriptors_ml17 = Descriptors.CalcMolDescriptors(molecule_ml17)
descriptors_ml18 = Descriptors.CalcMolDescriptors(molecule_ml18)
descriptors_ml19 = Descriptors.CalcMolDescriptors(molecule_ml19)
descriptors_ml20 = Descriptors.CalcMolDescriptors(molecule_ml20)
descriptors_ml21 = Descriptors.CalcMolDescriptors(molecule_ml21)
descriptors_ml22 = Descriptors.CalcMolDescriptors(molecule_ml22)
descriptors_ml23 = Descriptors.CalcMolDescriptors(molecule_ml23)
descriptors_ml24 = Descriptors.CalcMolDescriptors(molecule_ml24)
descriptors_ml25 = Descriptors.CalcMolDescriptors(molecule_ml25)
descriptors_ml26 = Descriptors.CalcMolDescriptors(molecule_ml26)

In [ ]:
molecular_descriptors_test_peptides = {
    'MolWt': [],
    'NHOHCount': [],
    'NOCount': [],
    'NumHAcceptors': [],
    'NumHDonors': [],
    'NumRotatableBonds': [],
    'MolLogP': [],
    'TPSA': []
}

def molecular_descriptors_calculation_peptides(descriptors_dict):
    for descriptor, value in descriptors_dict.items():
        if descriptor == 'MolWt':
            molecular_descriptors_test_peptides['MolWt'].append(value)
        elif descriptor == 'NHOHCount':
            molecular_descriptors_test_peptides['NHOHCount'].append(value)
        elif descriptor == 'NOCount':
            molecular_descriptors_test_peptides['NOCount'].append(value)
        elif descriptor == 'NumHAcceptors':
            molecular_descriptors_test_peptides['NumHAcceptors'].append(value)
        elif descriptor == 'NumHDonors':
            molecular_descriptors_test_peptides['NumHDonors'].append(value)
        elif descriptor == 'NumRotatableBonds':
            molecular_descriptors_test_peptides['NumRotatableBonds'].append(value)
        elif descriptor == 'MolLogP':
            molecular_descriptors_test_peptides['MolLogP'].append(value)
        elif descriptor == 'TPSA':
            molecular_descriptors_test_peptides['TPSA'].append(value)

In [ ]:
molecular_descriptors_calculation_peptides(descriptors_ml1)
molecular_descriptors_calculation_peptides(descriptors_ml2)
molecular_descriptors_calculation_peptides(descriptors_ml3)
molecular_descriptors_calculation_peptides(descriptors_ml4)
molecular_descriptors_calculation_peptides(descriptors_ml5)
molecular_descriptors_calculation_peptides(descriptors_ml6)
molecular_descriptors_calculation_peptides(descriptors_ml7)
molecular_descriptors_calculation_peptides(descriptors_ml8)
molecular_descriptors_calculation_peptides(descriptors_ml9)
molecular_descriptors_calculation_peptides(descriptors_ml10)
molecular_descriptors_calculation_peptides(descriptors_ml11)
molecular_descriptors_calculation_peptides(descriptors_ml12)
molecular_descriptors_calculation_peptides(descriptors_ml13)
molecular_descriptors_calculation_peptides(descriptors_ml14)
molecular_descriptors_calculation_peptides(descriptors_ml15)
molecular_descriptors_calculation_peptides(descriptors_ml16)
molecular_descriptors_calculation_peptides(descriptors_ml17)
molecular_descriptors_calculation_peptides(descriptors_ml18)
molecular_descriptors_calculation_peptides(descriptors_ml19)
molecular_descriptors_calculation_peptides(descriptors_ml20)
molecular_descriptors_calculation_peptides(descriptors_ml21)
molecular_descriptors_calculation_peptides(descriptors_ml22)
molecular_descriptors_calculation_peptides(descriptors_ml23)
molecular_descriptors_calculation_peptides(descriptors_ml24)
molecular_descriptors_calculation_peptides(descriptors_ml25)
molecular_descriptors_calculation_peptides(descriptors_ml26)

In [ ]:
test_dataset = pd.DataFrame.from_dict(molecular_descriptors_test_peptides, orient='index').T

In [ ]:
test_dataset = test_dataset.rename(columns={'MolWt': 'mw', 'MolLogP': 'xlogp', 'NumHDonors': 'hbonddonor', 'NumHAcceptors': 'hbondacc', 'TPSA': 'polararea', 'NumRotatableBonds': 'rotbonds'})

In [ ]:
test_dataset.drop(columns=['NHOHCount', 'NOCount'], inplace=True)

In [ ]:
test_dataset_lipinski = test_dataset[['mw', 'xlogp', 'hbonddonor', 'hbondacc']]

In [ ]:
test_dataset_muegge_bro = test_dataset[['mw', 'xlogp', 'hbonddonor', 'hbondacc', 'polararea', 'rotbonds']]

In [ ]:
attributes_lipinski_train, attributes_lipinski_test = train_test_split(dataset_with_violations[['mw', 'xlogp', 'hbonddonor', 'hbondacc']], test_size=0.2, random_state=42)

In [ ]:
target_lipinski_train, target_lipinski_test = train_test_split(dataset_with_violations[['nviolations_lipinski_rule']], test_size=0.2, random_state=42)

In [ ]:
attributes_muegge_bro_train, attributes_muegge_bro_test = train_test_split(dataset_with_violations[['mw', 'xlogp', 'hbonddonor', 'hbondacc', 'polararea', 'rotbonds']], test_size=0.2, random_state=42)

In [ ]:
target_muegge_train, target_muegge_test = train_test_split(dataset_with_violations[['nviolations_muegge_rule']], test_size=0.2, random_state=42)

In [ ]:
target_bro_train, target_bro_test = train_test_split(dataset_with_violations[['nviolations_bro_rule']], test_size=0.2, random_state=42)

In [ ]:
mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(max_iter=200, random_state=42))
])

param_grid_mlp = {
    'mlp__hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'mlp__activation': ['relu', 'tanh'],
    'mlp__alpha': [0.0001, 0.001]
}

grid_search_mlp = GridSearchCV(mlp_pipeline, param_grid_mlp, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search_mlp.fit(attributes_lipinski_train, target_lipinski_train.values.ravel())

print("Best parameters:", grid_search_mlp.best_params_)
print("Best CV score:", grid_search_mlp.best_score_)

predictions_mlp = grid_search_mlp.predict(attributes_lipinski_test)
print(classification_report(target_lipinski_test, predictions_mlp))

In [ ]:
mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(max_iter=200, random_state=42))
])

param_grid_mlp = {
    'mlp__hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'mlp__activation': ['relu', 'tanh'],
    'mlp__alpha': [0.0001, 0.001]
}

grid_search_mlp = GridSearchCV(mlp_pipeline, param_grid_mlp, cv=3, scoring='precision', n_jobs=-1, verbose=1)
grid_search_mlp.fit(attributes_lipinski_train, target_lipinski_train.values.ravel())

print("Best parameters:", grid_search_mlp.best_params_)
print("Best CV score:", grid_search_mlp.best_score_)

predictions_mlp = grid_search_mlp.predict(attributes_lipinski_test)
print(classification_report(target_lipinski_test, predictions_mlp))

In [ ]:
mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(max_iter=200, random_state=42))
])

param_grid_mlp = {
    'mlp__hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'mlp__activation': ['relu', 'tanh'],
    'mlp__alpha': [0.0001, 0.001]
}

grid_search_mlp = GridSearchCV(mlp_pipeline, param_grid_mlp, cv=3, scoring='recall', n_jobs=-1, verbose=1)
grid_search_mlp.fit(attributes_lipinski_train, target_lipinski_train.values.ravel())

print("Best parameters:", grid_search_mlp.best_params_)
print("Best CV score:", grid_search_mlp.best_score_)

predictions_mlp = grid_search_mlp.predict(attributes_lipinski_test)
print(classification_report(target_lipinski_test, predictions_mlp))

In [ ]:
mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(max_iter=200, random_state=42))
])

param_grid_mlp = {
    'mlp__hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'mlp__activation': ['relu', 'tanh'],
    'mlp__alpha': [0.0001, 0.001]
}

grid_search_mlp = GridSearchCV(mlp_pipeline, param_grid_mlp, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1)
grid_search_mlp.fit(attributes_lipinski_train, target_lipinski_train.values.ravel())

print("Best parameters:", grid_search_mlp.best_params_)
print("Best CV score:", grid_search_mlp.best_score_)

predictions_mlp = grid_search_mlp.predict(attributes_lipinski_test)
print(classification_report(target_lipinski_test, predictions_mlp))

In [ ]:
mlp_pipeline = Pipeline([('scaler', StandardScaler()), ('mlp', MLPClassifier(max_iter=200, random_state=42))])
param_grid_mlp = {
    'mlp__hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'mlp__activation': ['relu', 'tanh'],
    'mlp__alpha': [0.0001, 0.001]
}
grid_search_mlp_muegge = GridSearchCV(mlp_pipeline, param_grid_mlp, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search_mlp_muegge.fit(attributes_muegge_bro_train, target_muegge_train.values.ravel())
print("Best parameters (Muegge MLP):", grid_search_mlp_muegge.best_params_)
print("Best CV score:", grid_search_mlp_muegge.best_score_)
predictions = grid_search_mlp_muegge.predict(attributes_muegge_bro_test)
print(classification_report(target_muegge_test, predictions))

In [ ]:
mlp_pipeline = Pipeline([('scaler', StandardScaler()), ('mlp', MLPClassifier(max_iter=200, random_state=42))])
param_grid_mlp = {
    'mlp__hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'mlp__activation': ['relu', 'tanh'],
    'mlp__alpha': [0.0001, 0.001]
}
grid_search_mlp_muegge = GridSearchCV(mlp_pipeline, param_grid_mlp, cv=3, scoring='precision', n_jobs=-1, verbose=1)
grid_search_mlp_muegge.fit(attributes_muegge_bro_train, target_muegge_train.values.ravel())
print("Best parameters (Muegge MLP):", grid_search_mlp_muegge.best_params_)
print("Best CV score:", grid_search_mlp_muegge.best_score_)
predictions = grid_search_mlp_muegge.predict(attributes_muegge_bro_test)
print(classification_report(target_muegge_test, predictions))

In [ ]:
mlp_pipeline = Pipeline([('scaler', StandardScaler()), ('mlp', MLPClassifier(max_iter=200, random_state=42))])
param_grid_mlp = {
    'mlp__hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'mlp__activation': ['relu', 'tanh'],
    'mlp__alpha': [0.0001, 0.001]
}
grid_search_mlp_muegge = GridSearchCV(mlp_pipeline, param_grid_mlp, cv=3, scoring='recall', n_jobs=-1, verbose=1)
grid_search_mlp_muegge.fit(attributes_muegge_bro_train, target_muegge_train.values.ravel())
print("Best parameters (Muegge MLP):", grid_search_mlp_muegge.best_params_)
print("Best CV score:", grid_search_mlp_muegge.best_score_)
predictions = grid_search_mlp_muegge.predict(attributes_muegge_bro_test)
print(classification_report(target_muegge_test, predictions))

In [ ]:
mlp_pipeline = Pipeline([('scaler', StandardScaler()), ('mlp', MLPClassifier(max_iter=200, random_state=42))])
param_grid_mlp = {
    'mlp__hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'mlp__activation': ['relu', 'tanh'],
    'mlp__alpha': [0.0001, 0.001]
}
grid_search_mlp_muegge = GridSearchCV(mlp_pipeline, param_grid_mlp, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1)
grid_search_mlp_muegge.fit(attributes_muegge_bro_train, target_muegge_train.values.ravel())
print("Best parameters (Muegge MLP):", grid_search_mlp_muegge.best_params_)
print("Best CV score:", grid_search_mlp_muegge.best_score_)
predictions = grid_search_mlp_muegge.predict(attributes_muegge_bro_test)
print(classification_report(target_muegge_test, predictions))

In [ ]:
grid_search_mlp_bro = GridSearchCV(mlp_pipeline, param_grid_mlp, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search_mlp_bro.fit(attributes_muegge_bro_train, target_bro_train.values.ravel())
print("Best parameters (BRO MLP):", grid_search_mlp_bro.best_params_)
print("Best CV score:", grid_search_mlp_bro.best_score_)
predictions = grid_search_mlp_bro.predict(attributes_muegge_bro_test)
print(classification_report(target_bro_test, predictions))

In [ ]:
grid_search_mlp_bro = GridSearchCV(mlp_pipeline, param_grid_mlp, cv=3, scoring='precision', n_jobs=-1, verbose=1)
grid_search_mlp_bro.fit(attributes_muegge_bro_train, target_bro_train.values.ravel())
print("Best parameters (BRO MLP):", grid_search_mlp_bro.best_params_)
print("Best CV score:", grid_search_mlp_bro.best_score_)
predictions = grid_search_mlp_bro.predict(attributes_muegge_bro_test)
print(classification_report(target_bro_test, predictions))

In [ ]:
grid_search_mlp_bro = GridSearchCV(mlp_pipeline, param_grid_mlp, cv=3, scoring='recall', n_jobs=-1, verbose=1)
grid_search_mlp_bro.fit(attributes_muegge_bro_train, target_bro_train.values.ravel())
print("Best parameters (BRO MLP):", grid_search_mlp_bro.best_params_)
print("Best CV score:", grid_search_mlp_bro.best_score_)
predictions = grid_search_mlp_bro.predict(attributes_muegge_bro_test)
print(classification_report(target_bro_test, predictions))

In [ ]:
grid_search_mlp_bro = GridSearchCV(mlp_pipeline, param_grid_mlp, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1)
grid_search_mlp_bro.fit(attributes_muegge_bro_train, target_bro_train.values.ravel())
print("Best parameters (BRO MLP):", grid_search_mlp_bro.best_params_)
print("Best CV score:", grid_search_mlp_bro.best_score_)
predictions = grid_search_mlp_bro.predict(attributes_muegge_bro_test)
print(classification_report(target_bro_test, predictions))

In [ ]:
model_lipinski = MLPClassifier(activation='tanh', alpha=0.0001, hidden_layer_sizes=(50, 50))

In [ ]:
model_muegge = MLPClassifier(activation='tanh', alpha=0.0001, hidden_layer_sizes=(50, 50))

In [ ]:
model_bro = MLPClassifier(activation='tanh', alpha=0.001, hidden_layer_sizes=(50, 50))

In [ ]:
model_lipinski.fit(attributes_lipinski_train, target_lipinski_train)

In [ ]:
model_muegge.fit(attributes_muegge_bro_train, target_muegge_train)

In [ ]:
model_bro.fit(attributes_muegge_bro_train, target_bro_train)

In [ ]:
test_model_lipinski = model_lipinski.predict(attributes_lipinski_test)
test_model_muegge = model_muegge.predict(attributes_muegge_bro_test)
test_model_bro = model_bro.predict(attributes_muegge_bro_test)

In [ ]:
lipinski_accuracy = accuracy_score(target_lipinski_test,test_model_lipinski)
print(f"Accuracy of Lipinski rule predictions: {lipinski_accuracy}")

In [ ]:
lipinski_recall = recall_score(target_lipinski_test, test_model_lipinski, average='weighted')
print(f"Recall of Lipinski rule predictions: {lipinski_recall}")

In [ ]:
lipinski_precision = precision_score(target_lipinski_test, test_model_lipinski, average='weighted')
print(f"Precision of Lipinski rule predictions: {lipinski_precision}")

In [ ]:
lipinski_f1_score = f1_score(target_lipinski_test, test_model_lipinski, average='weighted')
print(f"F1 Score of Lipinski rule predictions: {lipinski_f1_score}")

In [ ]:
muegge_accuracy = accuracy_score(target_muegge_test, test_model_muegge)
print(f"Accuracy of Muegge rule predictions: {muegge_accuracy}")

In [ ]:
muegge_recall = recall_score(target_muegge_test, test_model_muegge, average='weighted')
print(f"Recall of Muegge rule predictions: {muegge_recall}")

In [ ]:
muegge_precision = precision_score(target_muegge_test, test_model_muegge, average='weighted')
print(f"Precision of Muegge rule predictions: {muegge_precision}")

In [ ]:
muegge_f1_score = f1_score(target_muegge_test, test_model_muegge, average='weighted')
print(f"F1 Score of Muegge rule predictions: {muegge_f1_score}")

In [ ]:
bro_accuracy = accuracy_score(target_bro_test, test_model_bro)
print(f"Accuracy of Broto rule predictions: {bro_accuracy}")

In [ ]:
bro_recall = recall_score(target_bro_test, test_model_bro, average='weighted')
print(f"Recall of Broto rule predictions: {bro_recall}")

In [ ]:
bro_precision = precision_score(target_bro_test, test_model_bro, average='weighted')
print(f"Precision of Broto rule predictions: {bro_precision}")

In [ ]:
bro_f1_score = f1_score(target_bro_test, test_model_bro, average='weighted')
print(f"F1 Score of Broto rule predictions: {bro_f1_score}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=300)

datasets = [
    (target_lipinski_test, test_model_lipinski, "Rule of Five (MLP)"),
    (target_muegge_test, test_model_muegge, "Muegge (MLP)"),
    (target_bro_test, test_model_bro, "Beyond Ro5 (MLP)")
]

for ax, (x, y, title) in zip(axes, datasets):
    ax.scatter(x, y, alpha=0.5, color='black')
    ax.set_xlabel("Test Values")
    ax.set_ylabel("Predicted Values")
    ax.set_title(title)
    ax.grid(True)

fig.tight_layout()

fig.savefig(
    "Scatter_Combined_MLP.jpg",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

In [ ]:
def plot_micro_roc(ax, y_true, y_score, title):
    classes = np.unique(y_true)
    y_bin = label_binarize(y_true, classes=classes)

    fpr, tpr, _ = roc_curve(
        y_bin.ravel(),
        y_score.ravel()
    )

    auc_micro = roc_auc_score(y_bin, y_score, average="micro")

    ax.plot(
        fpr,
        tpr,
        color="black",
        linewidth=2,
        label=f"AUC = {auc_micro:.2f}"
    )

    ax.plot(
        [0, 1], [0, 1],
        linestyle="--",
        color="gray",
        linewidth=1
    )

    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right")
    ax.grid(True)

    return auc_micro

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=300)

plot_micro_roc(
    axes[0],
    target_lipinski_test,
    model_lipinski.predict_proba(attributes_lipinski_test),
    "Rule of Five (MLP)"
)

plot_micro_roc(
    axes[1],
    target_muegge_test,
    model_muegge.predict_proba(attributes_muegge_bro_test),
    "Muegge (MLP)"
)

plot_micro_roc(
    axes[2],
    target_bro_test,
    model_bro.predict_proba(attributes_muegge_bro_test),
    "Beyond Ro5 (MLP)"
)

fig.tight_layout()

fig.savefig(
    "ROC_Combined_MLP.jpg",
    dpi=300,
    bbox_inches="tight"
)

plt.close()


In [ ]:
predictions_lipinski = model_lipinski.predict(test_dataset_lipinski)

In [ ]:
predictions_muegge = model_muegge.predict(test_dataset_muegge_bro)

In [ ]:
predictions_bro = model_bro.predict(test_dataset_muegge_bro)

In [ ]:
predicted_violations_peptides_lipinski = []

for predicted_value in predictions_lipinski:
    predicted_violations_peptides_lipinski.append(int(predicted_value))

In [ ]:
predicted_violations_peptides_muegge = []

for predicted_value in predictions_muegge:
    predicted_violations_peptides_muegge.append(int(predicted_value))

In [ ]:
predicted_violations_peptides_bro = []

for predicted_value in predictions_bro:
    predicted_violations_peptides_bro.append(int(predicted_value))

In [ ]:
lipinski_violations_dict = {'nviolations_lipinski_rule_mlp': predicted_violations_peptides_lipinski}
muegge_violations_dict = {'nviolations_muegge_rule_mlp': predicted_violations_peptides_muegge}
bro_violations_dict = {'nviolations_bro_rule_mlp': predicted_violations_peptides_bro}

In [ ]:
peptide_predictions_dataset_lipinski = pd.concat([pd.DataFrame(test_dataset_lipinski), pd.DataFrame(lipinski_violations_dict)], axis=1)

In [ ]:
peptide_predictions_dataset_lipinski

In [ ]:
peptide_predictions_dataset_muegge = pd.concat([pd.DataFrame(test_dataset_muegge_bro), pd.DataFrame(muegge_violations_dict)], axis=1)

In [ ]:
peptide_predictions_dataset_muegge

In [ ]:
peptide_predictions_dataset_bro = pd.concat([pd.DataFrame(test_dataset_muegge_bro), pd.DataFrame(bro_violations_dict)], axis=1)

In [ ]:
peptide_predictions_dataset_bro

In [ ]:
peptide_predictions_all = pd.concat([peptide_predictions_dataset_bro[['mw', 'xlogp', 'hbonddonor', 'hbondacc', 'polararea', 'rotbonds']], peptide_predictions_dataset_lipinski['nviolations_lipinski_rule_mlp'], peptide_predictions_dataset_muegge['nviolations_muegge_rule_mlp'], peptide_predictions_dataset_bro['nviolations_bro_rule_mlp']], axis=1)

In [ ]:
peptide_predictions_all